# AI Modified Workflow

For this workflow we use the following prompt to modify the `scale_model_size.py` notebook:

Rather than using containers can you download SST and run it on the bare metal?

---

Again, this one got stuck in an "Evaluating" step.  I reprompted it.  This one also took quite a bit of time to work through.

This is one task that I expected might be difficult but I was impressed with how well copilot nailed it.

There are a lot of things I like about how copilot adressed this one. I like:
  - that it updated comments and updated various places that referred to "containers" to now refer to the "sst source".
  - that it added parameters to the global params cell to specify where the sst repos lives and where we should install it.
  - that it added possible `DOWNLOAD_SST` and `BUILD_SST` steps into the `force` list and followed the precedent I set of checking these values at the top of a cell.
  - that it modified the `launch_and_log_sst` to launch bare-metal commands when the `image` parameter is `none`. Although it would be good to add a comment documenting that.
  - that it added a `sst_make_jobs` parameter that I can use to specify how much to parallelize the build.

The only critical thought I have is:

- That I'd prefer the download and build SST steps to be but done in separate cells. Having them in together in one individual cell seems quite lengthy and involved compared to other cells in the notebook.

As far as running it goes. When I ran the "Build benchmarks" step I get this error:

```
make: execvp: sst-register: Permission denied
```

This is an issue I run into when I do bare-metal builds and runs with sst manually.  I suspect this is some bug in NFS, if I, from the terminal, simply cd into `experiment/scale_model_size/sst-install/bin` and run `touch sst-register` I can unblock it.

There's no reasonable way copilot could have known this, and honestly I wouldn't want it to have modified the notebook to account for it.  But it does show that if there are quirks or oddities in your particular environment you may have to communicate those to copilot if you want it to improvise around those.

---
---
---

# Configuration

## Import workflows module

In [ ]:
# NOTE: Since copilot modified the workflows module I copied it into this new
# module and update the notebook to import it.
from workflows_new import *

## Global params

Users can modify these top-level parameters to alter the behavior of this workflow.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------------
# !!! DO NOT MODIFY THE CODE BELOW   !!!
# !!!  (Modify in the next section)  !!!
# ---------------------------------------------------------------------------------------------------------------------

# So that can you can maintain the defaults, we suggest you don't directly edit
# the parameters inline here but rather overwite values at the bottom of this
# cell.

# We'll store our workflow artifacts and benchmark results under the specified
# directory (it will be created if it doesn't already exist).
import os
if 'user_customExperimentsDir' in globals():
    baseDir = f'{user_customExperimentsDir}/scale_model_size'
else:
    baseDir=f'{os.getenv("HOME")}/workflows/scale_model_size'

# Bare-metal SST settings.
sst_repo = 'https://github.com/sstsimulator/sst-core.git'
sst_branch = 'master'
sst_src_dir = f'{baseDir}/sst-core'
sst_install_dir = f'{baseDir}/sst-install'
sst_configure_args = ''
sst_make_jobs = max(1, (os.cpu_count() or 4) - 1)

# The benchmark will be cloned from the specified repository. We assume the
# benchmark itself is in the 'benchmarkPath' directory within the repos.  We
# assume building the benchmark is a matter of running 'make' in that directoy.
benchmarkRepos='https://github.com/hpc-ai-adv-dev/sst-benchmarks.git'
benchmarkPath='phold'

# Run the benchmark on a single node, increasing the numbers of components with each trial
num_comps_per_trial  = [1_000_000, 2_000_000, 3_000_000, 4_000_000, 5_000_000]

# This command will be run prior to launching a job. The command will be run
# from within the benchmark directory and execution occurs within the worklaunch
# loop so it may be parameterized by the trial parameters if needed.
prestart_cmd_template = ''

# Indicates what arguments should be passed to sst and the benchmark each run 
# Note: {width} and {height} will be replaced with the appropriate values for
# each run, based on the number of nodes and components per node
sst_args_template   = '--print-timing-info=3 --parallel-load=SINGLE ./phold_dist.py'
bmark_args_template = '--width {width} --height {height}'

# Additional arguments to pass when launching jobs with srun. For example the
# partition name or --qos=high for higher priority in the queue.
additional_srun_args = ''

# Several of the setup steps will avoid rerunning if they have previously been run. Append to this
# list to indicate when you want to force a step to be reproduced.
#
# VALID VALUES ARE:
#   'ALL'
#   'DOWNLOAD_SST'
#   'BUILD_SST'
#   'DOWNLOAD_BENCHMARKS'
#   'BUILD_BENCHMARKS'      Note: we always rerun make, if this is set we will also run 'make clean' before rebuilding
force = []

# ---------------------------------------------------------------------------------------------------------------------
# Overwite parameters below this line to customize the workflow: 
# ---------------------------------------------------------------------------------------------------------------------

sst_branch = 'v15.1.2_Final'
num_comps_per_trial  = [1000, 2000, 3000, 4000, 5000]


## Environment

In [ ]:
set_workflow_log(f'{baseDir}/workflow.log')
run_cmd(f'e4s-cl profile edit --add-files {baseDir}')

## Download and build SST (bare metal)

In [ ]:
import shutil

_force_download = 'ALL' in force or 'DOWNLOAD_SST' in force
_force_build = 'ALL' in force or 'BUILD_SST' in force

if _force_download and os.path.exists(sst_src_dir):
    shutil.rmtree(sst_src_dir)

if not os.path.exists(sst_src_dir):
    run_cmd(f'git clone --depth 1 --branch {sst_branch} {sst_repo} {sst_src_dir}')
else:
    print(f'SST source already present at {sst_src_dir}, skipping clone.')

if _force_build and os.path.exists(sst_install_dir):
    shutil.rmtree(sst_install_dir)

sst_bin = f'{sst_install_dir}/bin/sst'
if not os.path.exists(sst_bin):
    cd(sst_src_dir)
    run_cmd('./autogen.sh')
    configure_cmd = f'./configure --prefix={sst_install_dir} {sst_configure_args}'.strip()
    run_cmd(configure_cmd)
    run_cmd(f'make -j{sst_make_jobs}')
    run_cmd('make install')
    cd(baseDir)
else:
    print(f'SST already built at {sst_install_dir}, skipping build.')

os.environ['PATH'] = f"{sst_install_dir}/bin:{os.environ.get('PATH', '')}"

## Download benchmarks

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_BENCHMARKS' in force

if not os.path.exists(f'benchmarks') or _force:
    run_cmd(f"git clone {benchmarkRepos} benchmarks")
    run_cmd(f"e4s-cl profile edit --add-files {baseDir}/benchmarks/{benchmarkPath}")
else:
    print(f"Benchmarks from {benchmarkRepos} have already been downloaded, skipping download.")

## Build benchmarks 

In [ ]:
_force = 'ALL' in force or 'BUILD_BENCHMARKS' in force

cd(f"{baseDir}/benchmarks/{benchmarkPath}")
run_cmd('touch sstsimulator.conf')
if _force:
    run_cmd('make clean')
run_cmd('make')
cd(baseDir)

# Run

## Start jobs

In [ ]:
import math, shutil, os

runDisplay = SafeDisplay(display_handle = display('', display_id="run_disp"))

# Setup directory to store results in
run_dir = f'{baseDir}/runs/'
if os.path.exists(run_dir):
    shutil.rmtree(run_dir)
os.makedirs(run_dir, exist_ok=True)

cd(f"{baseDir}/benchmarks/{benchmarkPath}")

# Deploy jobs
for approx_size in num_comps_per_trial:
    width  = int(math.sqrt(approx_size))
    height = width
    size = width*height

    if prestart_cmd_template is not None and prestart_cmd_template != '':
        run_cmd(prestart_cmd_template.format(width=width, height=height, size=size))

    full_sst_args_template = f'{sst_args_template} -- {bmark_args_template}'
    sst_args = full_sst_args_template.format(width=width, height=height, size=size)

    launch_and_log_sst(
        image        = None,
        srun_args    = f'-N 1 --job-name={benchmarkPath.lower()}_{size} {additional_srun_args}',
        sst_args     = sst_args,
        log_file     = f'{run_dir}/size_{size}',
        config_path  = f'{baseDir}/benchmarks/{benchmarkPath}/sstsimulator.conf',
        safe_display = runDisplay)

cd(f"{baseDir}")

## Watch squeue

In [ ]:
watch_queue_widget()

## Inspect results

In [ ]:
inspect_logs(f'{baseDir}/runs')

# Preprocess

In [ ]:
import os, glob

fullpath = f"{baseDir}/runs"
print(f'\n===== running extract under {fullpath} =====')
cd(fullpath)

data = extract_sst_output_in_files(sorted(glob.glob("size_*")))
csv_lines = convert_to_csv(data)
csv_name = f"{baseDir}/runs/results.csv"

with open(csv_name, 'w') as f:
    f.write('\n'.join(csv_lines))

if os.path.exists(csv_name):
    with open(csv_name, 'r') as f:
        print(f'\n===== {csv_name} =====')
        print(f.read())
else:
    print(f'\n===== {csv_name} (not created) =====')

# Plot

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

try:
    df = pd.read_csv(f"{baseDir}/runs/results.csv")
except FileNotFoundError as e:
    print(f'ERROR: File not found - {e.filename}')
    raise StopExecution()

fig = plt.figure()
ax = fig.add_subplot(111)

plot_value='total_duration'
ylabel = 'Total duration (secs)'

ax.scatter(x=df["Size"], y=df[plot_value], c='b', marker="s", label=f'SST 15.1.0')
ax.legend().remove()
plt.title(f'SST {benchmarkPath} single-node component scaling ({plot_value})')
plt.xlabel('Number of components')
plt.ylabel(ylabel)
plt.show()